# NER Model Comparison

Perbandingan komprehensif antara **Model Cahya** dan **Model Server** untuk Named Entity Recognition pada teks Bahasa Indonesia.

Evaluasi menggunakan dataset **NERGrit** yang merupakan benchmark standar untuk NER Indonesia.

# Setup

## Import Libraries

In [1]:
import json
import torch
from transformers import (
    BertTokenizer, BertConfig, BertPreTrainedModel, BertModel,
    AutoTokenizer, AutoModelForTokenClassification
)
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from typing import List
import plotly.graph_objects as go

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

/home/azril/miniconda3/envs/ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


# Data Generation

## Store Data

Load dataset NERGrit validation set yang sudah didownload dari IndoNLU benchmark.

In [2]:
with open('../data/nergrit/valid.json', 'r') as f:
    nergrit_valid = json.load(f)

print(f'Dataset: NERGrit validation set')
print(f'Sentences: {len(nergrit_valid)}')

TAG_MAPPING = {
    'B-PERSON': 'B-PER',
    'I-PERSON': 'I-PER',
    'B-PLACE': 'B-LOC',
    'I-PLACE': 'I-LOC',
    'B-ORGANISATION': 'B-ORG',
    'I-ORGANISATION': 'I-ORG',
    'O': 'O'
}

def normalize_tags(tags):
    return [TAG_MAPPING.get(t, t) for t in tags]

print(f"\nSample:")
ex = nergrit_valid[1]
print(f"  Tokens: {ex['tokens']}")
print(f"  Tags: {ex['ner_tags']}")

Dataset: NERGrit validation set
Sentences: 209

Sample:
  Tokens: ['Obama', 'belakangan', 'memicu', 'kontroversi', 'ketika', 'ia', 'meminta', 'Warren', 'untuk', 'memberikan', 'doa', 'berkat', 'pada', 'pelantikannya', 'sebagai', 'presiden', 'AS', 'pada', 'Januari', '2009', '.']
  Tags: ['B-PERSON', 'O', 'O', 'O', 'O', 'O', 'O', 'B-PERSON', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-PLACE', 'O', 'O', 'O', 'O']


# AI Credit Scoring

## Model 1: Cahya BERT

Model cahya/bert-base-indonesian-ner dari HuggingFace, dilatih pada dataset NERGrit dengan 19 entity types.

In [3]:
print('Loading Model Cahya...')
cahya_tokenizer = AutoTokenizer.from_pretrained("cahya/bert-base-indonesian-ner")
cahya_model = AutoModelForTokenClassification.from_pretrained("cahya/bert-base-indonesian-ner")
cahya_model.to(device)
cahya_model.eval()
cahya_id2label = cahya_model.config.id2label
print(f'Loaded. Labels: {len(cahya_id2label)}')

Loading Model Cahya...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 37346.93it/s]
BertForTokenClassification LOAD REPORT from: cahya/bert-base-indonesian-ner
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.pooler.dense.bias       | UNEXPECTED |  | 
bert.pooler.dense.weight     | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded. Labels: 39


In [4]:
def predict_cahya(tokens: List[str]) -> List[str]:
    """
    Predict menggunakan Model Cahya dengan proper token alignment.
    
    Args:
        tokens: List of word tokens
        
    Returns:
        List of BIO tags
    """
    encoding = cahya_tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    )
    
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    with torch.no_grad():
        outputs = cahya_model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
    
    predictions = torch.argmax(logits, dim=-1).squeeze().cpu().tolist()
    
    word_ids = encoding.word_ids()
    aligned_predictions = []
    previous_word_id = None
    
    for idx, word_id in enumerate(word_ids):
        if word_id is None:
            continue
        if word_id != previous_word_id:
            aligned_predictions.append(cahya_id2label[predictions[idx]])
        previous_word_id = word_id
    
    while len(aligned_predictions) < len(tokens):
        aligned_predictions.append('O')
    aligned_predictions = aligned_predictions[:len(tokens)]
    
    return aligned_predictions

## Model 2: Server NER

Model Server menggunakan IndoBERT dengan arsitektur MODPURPOSE custom. Label mapping diambil langsung dari production server.

In [5]:
idx2tag = {
    0: 'B-CRD', 1: 'B-DAT', 2: 'B-EVT', 3: 'B-FAC', 4: 'B-GPE', 5: 'B-LAN',
    6: 'B-LAW', 7: 'B-LOC', 8: 'B-MON', 9: 'B-NOR', 10: 'B-ORD', 11: 'B-ORG',
    12: 'B-PER', 13: 'B-PRC', 14: 'B-PRD', 15: 'B-QTY', 16: 'B-REG', 17: 'B-TIM',
    18: 'B-WOA', 19: 'I-CRD', 20: 'I-DAT', 21: 'I-EVT', 22: 'I-FAC', 23: 'I-GPE',
    24: 'I-LAN', 25: 'I-LAW', 26: 'I-LOC', 27: 'I-MON', 28: 'I-NOR', 29: 'I-ORD',
    30: 'I-ORG', 31: 'I-PER', 32: 'I-PRC', 33: 'I-PRD', 34: 'I-QTY', 35: 'I-REG',
    36: 'I-TIM', 37: 'I-WOA', 38: 'O'
}

class MODPURPOSE(BertPreTrainedModel):
    _tied_weights_keys = None
    
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.bert = BertModel(config)
        self.dropout = torch.nn.Dropout(config.hidden_dropout_prob)
        self.classifier = torch.nn.Linear(config.hidden_size, config.num_labels)

    def forward(self, input_ids=None, subword_to_word_ids=None, attention_mask=None,
                token_type_ids=None, position_ids=None, head_mask=None, inputs_embeds=None, labels=None):
        outputs = self.bert(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids,
                          position_ids=position_ids, head_mask=head_mask, inputs_embeds=inputs_embeds)
        
        sequence_output = outputs[0]
        max_seq_len = subword_to_word_ids.max() + 1
        word_latents = []
        for i in range(max_seq_len):
            mask = (subword_to_word_ids == i).unsqueeze(dim=-1)
            word_latents.append((sequence_output * mask).sum(dim=1) / mask.sum())
        sequence_output = torch.stack(word_latents, dim=1)
        
        sequence_output = self.dropout(sequence_output)
        sequence_output = self.classifier(sequence_output)
        
        return (sequence_output,) + outputs[2:]

print('Loading Model Server...')
server_tokenizer = BertTokenizer.from_pretrained('indolem/indobert-base-uncased')
server_config = BertConfig.from_pretrained('indolem/indobert-base-uncased')
server_config.num_labels = len(idx2tag)

server_model = MODPURPOSE(server_config)
bert_state = BertModel.from_pretrained('indolem/indobert-base-uncased').state_dict()
server_model.bert.load_state_dict(bert_state)
state_dict = torch.load('../models/server_ner_model.sav', map_location=device)
server_model.load_state_dict(state_dict, strict=False)
server_model.to(device)
server_model.eval()
print('Loaded')

Loading Model Server...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 51421.05it/s]
BertModel LOAD REPORT from: indolem/indobert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded


In [6]:
def word_subword_tokenize(sentence, tokenizer):
    subwords = [tokenizer.cls_token_id]
    subword_to_word_indices = [-1]
    for word_idx, word in enumerate(sentence):
        subword_list = tokenizer.encode(word, add_special_tokens=False)
        subword_to_word_indices += [word_idx for _ in range(len(subword_list))]
        subwords += subword_list
    subwords += [tokenizer.sep_token_id]
    subword_to_word_indices += [-1]
    return subwords, subword_to_word_indices

def predict_server(tokens: List[str]) -> List[str]:
    """
    Predict menggunakan Model Server dengan production method.
    
    Args:
        tokens: List of word tokens
        
    Returns:
        List of BIO tags
    """
    subwords, subword_to_word_indices = word_subword_tokenize(tokens, server_tokenizer)
    
    subwords_tensor = torch.LongTensor(subwords).view(1, -1).to(device)
    sw2w_tensor = torch.LongTensor(subword_to_word_indices).view(1, -1).to(device)
    
    with torch.no_grad():
        logits = server_model(subwords_tensor, sw2w_tensor)[0]
    
    preds = torch.topk(logits, k=1, dim=-1)[1].squeeze().cpu().numpy()
    
    if len(preds.shape) == 0:
        preds = [preds.item()]
    
    labels = [idx2tag[preds[i]] for i in range(len(preds))]
    return labels

## Run Evaluation

In [7]:
print('Evaluating all sentences...')
cahya_true_all = []
cahya_pred_all = []
server_true_all = []
server_pred_all = []

for i, ex in enumerate(nergrit_valid):
    if (i+1) % 50 == 0:
        print(f'  Processing {i+1}/{len(nergrit_valid)}...')
    
    tokens = ex['tokens']
    true_tags = normalize_tags(ex['ner_tags'])
    
    cahya_pred = predict_cahya(tokens)
    server_pred = predict_server(tokens)
    
    cahya_pred_normalized = [tag.replace('GPE', 'LOC') for tag in cahya_pred]
    server_pred_normalized = [tag.replace('GPE', 'LOC') for tag in server_pred]
    
    cahya_true_all.append(true_tags)
    cahya_pred_all.append(cahya_pred_normalized)
    server_true_all.append(true_tags)
    server_pred_all.append(server_pred_normalized)

print('Evaluation complete!')

Evaluating all sentences...
  Processing 50/209...


Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/home/azril/miniconda3/envs/ml/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "/home/azril/miniconda3/envs/ml/lib/python3.11/threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "/home/azril/miniconda3/envs/ml/lib/python3.11/site-packages/transformers/safetensors_conversion.py", line 117, in auto_conversion
    raise e
  File "/home/azril/miniconda3/envs/ml/lib/python3.11/site-packages/transformers/safetensors_conversion.py", line 96, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/azril/miniconda3/envs/ml/lib/python3.11/site-packages/transformers/safetensors_conversion.py", line 72, in get_conversion_pr_reference
    spawn_conversion(token, private, model_id)

  Processing 100/209...
  Processing 150/209...
  Processing 200/209...
Evaluation complete!


## Results

In [8]:
print('=' * 70)
print('MODEL CAHYA')
print('=' * 70)
cahya_f1 = f1_score(cahya_true_all, cahya_pred_all)
cahya_precision = precision_score(cahya_true_all, cahya_pred_all)
cahya_recall = recall_score(cahya_true_all, cahya_pred_all)
print(f'Precision: {cahya_precision:.4f}')
print(f'Recall:    {cahya_recall:.4f}')
print(f'F1-Score:  {cahya_f1:.4f}')
print('\nDetailed Report:')
print(classification_report(cahya_true_all, cahya_pred_all))

MODEL CAHYA
Precision: 0.4035
Recall:    0.7009
F1-Score:  0.5121

Detailed Report:
              precision    recall  f1-score   support

         CRD       0.00      0.00      0.00         0
         DAT       0.00      0.00      0.00         0
         EVT       0.00      0.00      0.00         0
         FAC       0.00      0.00      0.00         0
         LAN       0.00      0.00      0.00         0
         LAW       0.00      0.00      0.00         0
         LOC       0.78      0.70      0.73       328
         MON       0.00      0.00      0.00         0
         NOR       0.00      0.00      0.00         0
         ORD       0.00      0.00      0.00         0
         ORG       0.42      0.61      0.50       121
         PER       0.76      0.76      0.76       213
         PRD       0.00      0.00      0.00         0
         QTY       0.00      0.00      0.00         0
         REG       0.00      0.00      0.00         0
         WOA       0.00      0.00      0.00        

/home/azril/miniconda3/envs/ml/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [9]:
print('=' * 70)
print('MODEL SERVER')
print('=' * 70)
server_f1 = f1_score(server_true_all, server_pred_all)
server_precision = precision_score(server_true_all, server_pred_all)
server_recall = recall_score(server_true_all, server_pred_all)
print(f'Precision: {server_precision:.4f}')
print(f'Recall:    {server_recall:.4f}')
print(f'F1-Score:  {server_f1:.4f}')
print('\nDetailed Report:')
print(classification_report(server_true_all, server_pred_all))

MODEL SERVER
Precision: 0.4095
Recall:    0.7281
F1-Score:  0.5242

Detailed Report:
              precision    recall  f1-score   support

         CRD       0.00      0.00      0.00         0
         DAT       0.00      0.00      0.00         0
         EVT       0.00      0.00      0.00         0
         FAC       0.00      0.00      0.00         0
         LAN       0.00      0.00      0.00         0
         LAW       0.00      0.00      0.00         0
         LOC       0.79      0.72      0.75       328
         MON       0.00      0.00      0.00         0
         NOR       0.00      0.00      0.00         0
         ORD       0.00      0.00      0.00         0
         ORG       0.52      0.62      0.56       121
         PER       0.81      0.81      0.81       213
         PRD       0.00      0.00      0.00         0
         QTY       0.00      0.00      0.00         0
         REG       0.00      0.00      0.00         0
         WOA       0.00      0.00      0.00       

## Filtered Evaluation

Evaluasi di atas menggunakan semua 19 entity types yang diprediksi model. Namun dataset NERGrit hanya memiliki 3 entity types (PER, LOC, ORG). Akibatnya, prediksi yang benar secara semantik (seperti DATE, EVENT, PRODUCT) dihitung sebagai error.

Untuk evaluasi yang lebih fair, kita filter prediksi agar hanya fokus pada entity types yang ada di dataset.

In [10]:
def filter_to_core_entities(tags):
    """
    Filter prediksi agar hanya menyimpan PER, LOC, ORG (dan GPE→LOC).
    Entity types lain (DAT, EVT, PRD, dll) diubah menjadi O.
    """
    result = []
    for t in tags:
        if any(x in t for x in ['PER', 'LOC', 'ORG', 'GPE']):
            result.append(t.replace('GPE', 'LOC'))
        else:
            result.append('O')
    return result

cahya_pred_filtered = [filter_to_core_entities(seq) for seq in cahya_pred_all]
server_pred_filtered = [filter_to_core_entities(seq) for seq in server_pred_all]

print('=' * 70)
print('FILTERED EVALUATION (hanya PER, LOC, ORG)')
print('=' * 70)

cahya_f1_filtered = f1_score(cahya_true_all, cahya_pred_filtered)
cahya_precision_filtered = precision_score(cahya_true_all, cahya_pred_filtered)
cahya_recall_filtered = recall_score(cahya_true_all, cahya_pred_filtered)

server_f1_filtered = f1_score(server_true_all, server_pred_filtered)
server_precision_filtered = precision_score(server_true_all, server_pred_filtered)
server_recall_filtered = recall_score(server_true_all, server_pred_filtered)

print(f"\n{'Model':<15} {'Precision':>12} {'Recall':>12} {'F1-Score':>12}")
print('-' * 55)
print(f"{'Cahya':<15} {cahya_precision_filtered:>12.4f} {cahya_recall_filtered:>12.4f} {cahya_f1_filtered:>12.4f}")
print(f"{'Server':<15} {server_precision_filtered:>12.4f} {server_recall_filtered:>12.4f} {server_f1_filtered:>12.4f}")
print('-' * 55)
winner = 'Cahya' if cahya_f1_filtered > server_f1_filtered else 'Server'
diff = abs(cahya_f1_filtered - server_f1_filtered) * 100
print(f"Winner: {winner} (+{diff:.2f}% F1)")

print('\n' + '=' * 70)
print('PERBANDINGAN: Standard vs Filtered')
print('=' * 70)
print(f"\n{'Metode':<20} {'Cahya F1':>12} {'Server F1':>12}")
print('-' * 45)
print(f"{'Standard':<20} {cahya_f1:>12.4f} {server_f1:>12.4f}")
print(f"{'Filtered':<20} {cahya_f1_filtered:>12.4f} {server_f1_filtered:>12.4f}")
print(f"{'Improvement':<20} {(cahya_f1_filtered-cahya_f1)*100:>+11.1f}% {(server_f1_filtered-server_f1)*100:>+11.1f}%")

FILTERED EVALUATION (hanya PER, LOC, ORG)

Model              Precision       Recall     F1-Score
-------------------------------------------------------
Cahya                 0.6804       0.7009       0.6905
Server                0.7359       0.7281       0.7320
-------------------------------------------------------
Winner: Server (+4.15% F1)

PERBANDINGAN: Standard vs Filtered

Metode                   Cahya F1    Server F1
---------------------------------------------
Standard                   0.5121       0.5242
Filtered                   0.6905       0.7320
Improvement                +17.8%       +20.8%


## Comparison Visualization

In [11]:
fig = go.Figure(data=[
    go.Bar(name='Model Cahya', x=['Precision', 'Recall', 'F1-Score'],
           y=[cahya_precision, cahya_recall, cahya_f1], marker_color='steelblue'),
    go.Bar(name='Model Server', x=['Precision', 'Recall', 'F1-Score'],
           y=[server_precision, server_recall, server_f1], marker_color='darkorange')
])

fig.update_layout(
    title='Model Comparison: Cahya vs Server',
    xaxis_title='Metric',
    yaxis_title='Score',
    barmode='group',
    yaxis=dict(range=[0, 1])
)

fig.show()

In [12]:
print('=' * 70)
print('COMPARISON SUMMARY')
print('=' * 70)
print(f"{'Metric':<15} {'Cahya':>10} {'Server':>10} {'Winner':>10}")
print('-' * 45)
print(f"{'Precision':<15} {cahya_precision:>10.4f} {server_precision:>10.4f} {'Cahya' if cahya_precision > server_precision else 'Server':>10}")
print(f"{'Recall':<15} {cahya_recall:>10.4f} {server_recall:>10.4f} {'Cahya' if cahya_recall > server_recall else 'Server':>10}")
print(f"{'F1-Score':<15} {cahya_f1:>10.4f} {server_f1:>10.4f} {'Cahya' if cahya_f1 > server_f1 else 'Server':>10}")
print('=' * 70)

COMPARISON SUMMARY
Metric               Cahya     Server     Winner
---------------------------------------------
Precision           0.4035     0.4095     Server
Recall              0.7009     0.7281     Server
F1-Score            0.5121     0.5242     Server


## Filtered Evaluation Comparison


In [16]:
fig_filtered = go.Figure(data=[
    go.Bar(name='Model Cahya', x=['Precision', 'Recall', 'F1-Score'],
           y=[cahya_precision_filtered, cahya_recall_filtered, cahya_f1_filtered], marker_color='steelblue'),
    go.Bar(name='Model Server', x=['Precision', 'Recall', 'F1-Score'],
           y=[server_precision_filtered, server_recall_filtered, server_f1_filtered], marker_color='darkorange')
])

fig_filtered.update_layout(
    title='Model Comparison (Filtered Evaluation: PER, LOC, ORG)',
    xaxis_title='Metric',
    yaxis_title='Score',
    barmode='group',
    yaxis=dict(range=[0, 1])
)

fig_filtered.show()


## Per-Entity Performance

Perbandingan performa model untuk setiap jenis entity (PER, LOC, ORG) secara individual.


In [17]:
def get_per_entity_metrics(true_tags_all, pred_tags_all, entities):
    """
    Menghitung precision, recall, dan F1-score per entity type.
    """
    results = {}
    
    for entity in entities:
        true_entity = []
        pred_entity = []
        
        for true_seq, pred_seq in zip(true_tags_all, pred_tags_all):
            true_entity_seq = []
            pred_entity_seq = []
            
            for true, pred in zip(true_seq, pred_seq):
                true_entity_seq.append(true if entity in true else 'O')
                pred_entity_seq.append(pred if entity in pred else 'O')
            
            true_entity.append(true_entity_seq)
            pred_entity.append(pred_entity_seq)
        
        if any(tag != 'O' for seq in true_entity for tag in seq):
            prec = precision_score(true_entity, pred_entity)
            rec = recall_score(true_entity, pred_entity)
            f1 = f1_score(true_entity, pred_entity)
        else:
            prec = rec = f1 = 0.0
        
        results[entity] = {'precision': prec, 'recall': rec, 'f1': f1}
    
    return results

cahya_per_entity = get_per_entity_metrics(cahya_true_all, cahya_pred_filtered, ['PER', 'LOC', 'ORG'])
server_per_entity = get_per_entity_metrics(server_true_all, server_pred_filtered, ['PER', 'LOC', 'ORG'])

entities = list(cahya_per_entity.keys())
cahya_f1_per_entity = [cahya_per_entity[e]['f1'] for e in entities]
server_f1_per_entity = [server_per_entity[e]['f1'] for e in entities]

fig_per_entity = go.Figure(data=[
    go.Bar(name='Model Cahya', x=entities, y=cahya_f1_per_entity, marker_color='steelblue'),
    go.Bar(name='Model Server', x=entities, y=server_f1_per_entity, marker_color='darkorange')
])

fig_per_entity.update_layout(
    title='Per-Entity F1-Score Comparison',
    xaxis_title='Entity Type',
    yaxis_title='F1-Score',
    barmode='group',
    yaxis=dict(range=[0, 1])
)

fig_per_entity.show()

print("\nPer-Entity Detailed Metrics:")
print("=" * 70)
print(f"{'Entity':<10} {'Model':<12} {'Precision':>12} {'Recall':>12} {'F1-Score':>12}")
print("-" * 70)

for entity in entities:
    cahya_m = cahya_per_entity[entity]
    server_m = server_per_entity[entity]
    
    print(f"{entity:<10} {'Cahya':<12} {cahya_m['precision']:>12.4f} {cahya_m['recall']:>12.4f} {cahya_m['f1']:>12.4f}")
    print(f"{'':<10} {'Server':<12} {server_m['precision']:>12.4f} {server_m['recall']:>12.4f} {server_m['f1']:>12.4f}")
    print("-" * 70)



Per-Entity Detailed Metrics:
Entity     Model           Precision       Recall     F1-Score
----------------------------------------------------------------------
PER        Cahya              0.7606       0.7606       0.7606
           Server             0.8075       0.8075       0.8075
----------------------------------------------------------------------
LOC        Cahya              0.7782       0.6951       0.7343
           Server             0.7912       0.7165       0.7520
----------------------------------------------------------------------
ORG        Cahya              0.4205       0.6116       0.4983
           Server             0.5172       0.6198       0.5639
----------------------------------------------------------------------


# Error Analysis

Analisis mendalam terhadap kesalahan prediksi kedua model untuk memahami karakteristik masing-masing.

## Qualitative Examples

Contoh konkret prediksi dari kedua model pada beberapa kalimat uji.

In [18]:
def show_prediction_comparison(idx, tokens, true_tags, cahya_tags, server_tags):
    """
    Menampilkan perbandingan prediksi dalam format yang mudah dibaca.
    Highlight perbedaan antara ground truth dan prediksi.
    """
    print(f"\n{'='*70}")
    print(f"Kalimat #{idx+1}")
    print(f"{'='*70}")
    print(f"Text: {' '.join(tokens)}\n")
    
    print(f"{'Token':<20} {'Ground Truth':<15} {'Cahya':<15} {'Server':<15}")
    print('-' * 65)
    
    for tok, true, cahya, server in zip(tokens, true_tags, cahya_tags, server_tags):
        cahya_mark = '✓' if cahya == true else '✗'
        server_mark = '✓' if server == true else '✗'
        
        if true != 'O' or cahya != 'O' or server != 'O':
            print(f"{tok:<20} {true:<15} {cahya:<12}{cahya_mark:>3} {server:<12}{server_mark:>3}")

sample_indices = [0, 1, 5, 10, 50]

for idx in sample_indices:
    if idx < len(nergrit_valid):
        tokens = nergrit_valid[idx]['tokens']
        true_tags = normalize_tags(nergrit_valid[idx]['ner_tags'])
        
        cahya_tags = predict_cahya(tokens)[:len(tokens)]
        server_tags = predict_server(tokens)[:len(tokens)]
        
        while len(cahya_tags) < len(tokens):
            cahya_tags.append('O')
        while len(server_tags) < len(tokens):
            server_tags.append('O')
        
        show_prediction_comparison(idx, tokens, true_tags, cahya_tags, server_tags)


Kalimat #1
Text: Walaupun demikian , sumber - sumber lain meragukan catatan rekor " Guinness Book " , dan menyatakan Bhosle lebih banyak merekam lagu daripada Mangeshkar .

Token                Ground Truth    Cahya           Server         
-----------------------------------------------------------------
Guinness             O               B-PRD         ✗ B-ORG         ✗
Book                 O               I-PRD         ✗ I-ORG         ✗
Bhosle               B-PER           B-PER         ✓ B-ORG         ✗
Mangeshkar           B-PER           B-PER         ✓ B-PER         ✓

Kalimat #2
Text: Obama belakangan memicu kontroversi ketika ia meminta Warren untuk memberikan doa berkat pada pelantikannya sebagai presiden AS pada Januari 2009 .

Token                Ground Truth    Cahya           Server         
-----------------------------------------------------------------
Obama                B-PER           B-PER         ✓ B-PER         ✓
Warren               B-PER           B-PER  

## Error Pattern Analysis

Mengkategorikan jenis kesalahan yang dibuat oleh masing-masing model.

In [19]:
def analyze_errors(true_tags_all, pred_tags_all, model_name):
    """
    Menganalisis pattern kesalahan model.
    """
    false_positive = 0  
    false_negative = 0  
    wrong_type = 0      
    correct = 0
    
    for true_seq, pred_seq in zip(true_tags_all, pred_tags_all):
        for true, pred in zip(true_seq, pred_seq):
            true_is_entity = true != 'O'
            pred_is_entity = pred != 'O'
            
            if true == pred:
                correct += 1
            elif not true_is_entity and pred_is_entity:
                false_positive += 1
            elif true_is_entity and not pred_is_entity:
                false_negative += 1
            else:
                wrong_type += 1
    
    total_errors = false_positive + false_negative + wrong_type
    
    print(f"\n{model_name} Error Analysis:")
    print(f"  Correct predictions: {correct}")
    print(f"  Total errors: {total_errors}")
    if total_errors > 0:
        print(f"    - False Positive (prediksi entity padahal bukan): {false_positive} ({false_positive/total_errors*100:.1f}%)")
        print(f"    - False Negative (miss entity): {false_negative} ({false_negative/total_errors*100:.1f}%)")
        print(f"    - Wrong Type (entity benar tapi tipe salah): {wrong_type} ({wrong_type/total_errors*100:.1f}%)")

analyze_errors(cahya_true_all, cahya_pred_all, 'Model Cahya')
analyze_errors(server_true_all, server_pred_all, 'Model Server')


Model Cahya Error Analysis:
  Correct predictions: 5606
  Total errors: 1377
    - False Positive (prediksi entity padahal bukan): 1078 (78.3%)
    - False Negative (miss entity): 38 (2.8%)
    - Wrong Type (entity benar tapi tipe salah): 261 (19.0%)

Model Server Error Analysis:
  Correct predictions: 5579
  Total errors: 1404
    - False Positive (prediksi entity padahal bukan): 1102 (78.5%)
    - False Negative (miss entity): 16 (1.1%)
    - Wrong Type (entity benar tapi tipe salah): 286 (20.4%)


## Speed Benchmarking

Perbandingan kecepatan inference kedua model.

In [20]:
import time

test_sentences = [nergrit_valid[i]['tokens'] for i in range(min(50, len(nergrit_valid)))]

start = time.time()
for tokens in test_sentences:
    _ = predict_cahya(tokens)
cahya_time = time.time() - start

start = time.time()
for tokens in test_sentences:
    _ = predict_server(tokens)
server_time = time.time() - start

print(f"Speed Benchmark (50 sentences):")
print(f"  Model Cahya:  {cahya_time:.2f}s ({cahya_time/50*1000:.1f}ms/sentence)")
print(f"  Model Server: {server_time:.2f}s ({server_time/50*1000:.1f}ms/sentence)")
print(f"  Winner: {'Cahya' if cahya_time < server_time else 'Server'} ({abs(cahya_time-server_time)/max(cahya_time,server_time)*100:.1f}% faster)")

Speed Benchmark (50 sentences):
  Model Cahya:  0.28s (5.7ms/sentence)
  Model Server: 0.32s (6.5ms/sentence)
  Winner: Cahya (12.6% faster)


# Kesimpulan

## Hasil Evaluasi

Evaluasi dilakukan pada **NERGrit validation set** (benchmark standar NER Indonesia) dengan 209 kalimat dan 662 entitas.

## Metodologi Evaluasi

Karena kedua model mampu mengenali 19 entity types, sedangkan dataset hanya memiliki 3 types (PER, LOC, ORG), kami menggunakan **Filtered Evaluation** yang hanya mengevaluasi entity types yang ada di dataset.

## Hasil Perbandingan

### Standard Evaluation (semua entity types)
| Model | Precision | Recall | F1-Score |
|-------|-----------|--------|----------|
| Cahya | 0.40 | 0.70 | 0.51 |
| Server | 0.41 | 0.73 | 0.52 |

### Filtered Evaluation (hanya PER, LOC, ORG)
| Model | Precision | Recall | F1-Score |
|-------|-----------|--------|----------|
| Cahya | 0.68 | 0.70 | 0.69 |
| Server | 0.71 | 0.73 | 0.72 |

## Per-Entity Performance
- **PER (Person)**: Server lebih baik (0.81 vs 0.76)
- **LOC (Location)**: Server sedikit lebih baik (0.75 vs 0.73)
- **ORG (Organization)**: Server lebih baik (0.56 vs 0.50)

## Catatan Penting

### Mengapa Filtered Evaluation lebih fair?
Model memprediksi DATE, EVENT, PRODUCT dll dengan benar secara semantik, tapi dihitung sebagai error karena ground truth tidak memiliki label tersebut. Filtered evaluation menghilangkan bias ini.

### Catatan Teknis
- Kedua model menggunakan label set identik (19 entity types)
- Model Cahya: BERT standar dari HuggingFace
- Model Server: IndoBERT dengan arsitektur MODPURPOSE (subword-to-word aggregation)

## Rekomendasi

**Model Server** menunjukkan performa yang lebih baik pada semua entity types. Dengan F1-score 0.72 (filtered), Model Server dapat direkomendasikan untuk production. Namun Model Cahya dengan F1=0.69 juga merupakan pilihan valid jika kemudahan deployment menjadi prioritas.